# GreenCampus AI – AI Decision Support and Recommendations

## 1. Introduction

This notebook converts AI anomaly results into simple,
human-readable recommendations.

The recommendations are intended to support investigation and
decision-making by campus administrators and maintenance staff.

#### Loading AI results

In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv("C:/Users/Urja Chauhan/PycharmProjects/PythonProject/campus_sustainability_ai_results.csv")

df["Date"] = pd.to_datetime(df["Date"])

df.head()

,Date,Building,Building_Type,Occupancy,Electricity_kWh,Water_Litres,Waste_kg,Temperature_C,Working_Day,Event_Day,Anomaly_Prediction,Anomaly,Anomaly_Score,AI_Insight
0,2025-07-01,Block A,Academic,444,1437.09,8384.17,47.61,28.0,Yes,No,1,Normal,0.099599,Resource consumption appears within the detect...
1,2025-07-01,Block B,Academic,269,1091.78,4583.04,23.77,28.0,Yes,No,1,Normal,0.141118,Resource consumption appears within the detect...
2,2025-07-01,Block C,Laboratory,186,707.71,3556.96,22.42,28.0,Yes,No,1,Normal,0.138075,Resource consumption appears within the detect...
3,2025-07-01,Library,Library,230,760.70,4039.62,24.38,28.0,Yes,No,1,Normal,0.131434,Resource consumption appears within the detect...
4,2025-07-01,Hostel,Hostel,555,1679.32,9611.07,54.21,28.0,Yes,No,1,Normal,0.055242,Resource consumption appears within the detect...


## 2. Resource Consumption Thresholds

In [2]:
electricity_threshold = df["Electricity_kWh"].quantile(0.95)
water_threshold = df["Water_Litres"].quantile(0.95)
waste_threshold = df["Waste_kg"].quantile(0.95)

print("Electricity threshold: ", round(electricity_threshold,2))
print("Water threshold: ", round(water_threshold, 2))
print("Waste threshold: ", round(waste_threshold, 2))

Electricity threshold:  1798.71
Water threshold:  9446.3
Waste threshold:  53.34


## 3. Recommendation Logic

#### Create recommendation function

In [3]:
def generate_recommendation(row):

    if row["Anomaly"] == "Normal":
        return "No unusual pattern detected. Continue routine monitoring."

    if row["Electricity_kWh"] >= electricity_threshold:
        return ("Unusually high electricity consumption detected."
                "Investigate equipment usage, cooling demand and operating hours, "
                "or possible energy wastage.")

    elif row["Water_Litres"] >= water_threshold:
        return ("Unusually high water consumption detected. "
                "Investigate possible leakage, meter issues, "
                "or unusual water usage.")

    elif row["Waste_kg"] >= waste_threshold:
        return ("Unusually high waste generation detected. "
                "Review event activity, waste segregation, "
                "and disposal practices.")

    else:
        return ("Unusual overall resource-consumption pattern detected. "
                "Further investigation is recommended.")

## 4. Generate AI Recommendations

In [4]:
df["AI_Recommendation"] = df.apply(generate_recommendation, axis=1)

In [5]:
df[["Date", "Building", "Anomaly", "Electricity_kWh", "Water_Litres", "Waste_kg", "AI_Recommendation"]].head(20)

,Date,Building,Anomaly,Electricity_kWh,Water_Litres,Waste_kg,AI_Recommendation
0,2025-07-01,Block A,Normal,1553.96,8376.81,38.92,No unusual pattern detected. Continue routine ...
1,2025-07-01,Block B,Normal,1071.22,6642.88,29.01,No unusual pattern detected. Continue routine ...
2,2025-07-01,Block C,Normal,887.68,3155.65,21.21,No unusual pattern detected. Continue routine ...
3,2025-07-01,Library,Normal,990.07,5183.24,24.40,No unusual pattern detected. Continue routine ...
4,2025-07-01,Hostel,Normal,1874.57,8503.42,49.87,No unusual pattern detected. Continue routine ...
5,2025-07-02,Block A,Normal,1608.19,7372.86,37.09,No unusual pattern detected. Continue routine ...
6,2025-07-02,Block B,Normal,1292.00,5614.62,31.46,No unusual pattern detected. Continue routine ...
7,2025-07-02,Block C,Normal,767.89,3358.27,9.23,No unusual pattern detected. Continue routine ...
8,2025-07-02,Library,Normal,1005.63,4028.19,24.68,No unusual pattern detected. Continue routine ...
9,2025-07-02,Hostel,Normal,1708.97,8094.70,47.87,No unusual pattern detected. Continue routine ...


## 5. Review of Anomalous Cases

In [6]:
anomalies = df[df["Anomaly"] == "Anomaly"]

anomalies[["Date", "Building", "Electricity_kWh", "Water_Litres", "Waste_kg", "AI_Recommendation"]].head(20)

,Date,Building,Electricity_kWh,Water_Litres,Waste_kg,AI_Recommendation
79,2025-07-16,Hostel,1854.50,10095.70,81.11,Unusually high electricity consumption detecte...
154,2025-07-31,Hostel,2017.45,8911.06,51.29,Unusually high electricity consumption detecte...
249,2025-08-19,Hostel,1801.08,10354.48,60.02,Unusually high electricity consumption detecte...
259,2025-08-21,Hostel,1922.63,9844.31,58.27,Unusually high electricity consumption detecte...
290,2025-08-28,Block A,1423.51,9314.49,63.68,Unusually high waste generation detected. Revi...
294,2025-08-28,Hostel,1847.84,9732.12,73.00,Unusually high electricity consumption detecte...
319,2025-09-02,Hostel,1959.72,12850.21,84.27,Unusually high electricity consumption detecte...
364,2025-09-11,Hostel,1748.32,10476.07,64.73,Unusually high water consumption detected. Inv...
389,2025-09-16,Hostel,1769.45,10607.90,56.93,Unusually high water consumption detected. Inv...
539,2025-10-16,Hostel,1945.32,11364.97,82.45,Unusually high electricity consumption detecte...


## 6. Recommendation Categories

In [7]:
recommendation_counts = (anomalies["AI_Recommendation"].value_counts())
print(recommendation_counts)

AI_Recommendation
Unusually high electricity consumption detected.Investigate equipment usage, cooling demand and operating hours, or possible energy wastage.    22
Unusually high water consumption detected. Investigate possible leakage, meter issues, or unusual water usage.                                  18
Unusual overall resource-consumption pattern detected. Further investigation is recommended.                                                    14
Unusually high waste generation detected. Review event activity, waste segregation, and disposal practices.                                      1
Name: count, dtype: int64


## 7. Save Final Recommendation Dataset

In [8]:
df.to_csv("C:/Users/Urja Chauhan/PycharmProjects/PythonProject/campus_sustainability_ai_recommendations.csv", index=False)

print("Recommendation file saved successfully!")

Recommendation file saved successfully!


## 8. Human Review and Responsible AI

The recommendations identify unusual resource-consumption patterns
and suggest areas for investigation.

They do not confirm the exact cause of an issue.

For example, unusually high water consumption may indicate the need
to investigate possible leakage, meter issues, or unusual usage.

Final decisions should be made by appropriate human users after
checking the actual campus situation.